# Model-Free Reinforcement Learning: SARSA, Q-learning, Expected SARSA

This notebook implements three model-free RL algorithms and compares
them on the **Cliff Walking** task.

**Cliff Walking** is a 4×12 gridworld (48 states, 4 actions).
The agent starts at the bottom-left and must reach the bottom-right goal.
The bottom row (except start and goal) is a "cliff" — stepping on it
gives reward −100 and resets the agent to the start.
All other transitions give reward −1.

| Algorithm | Update target |
|-----------|--------------|
| SARSA | $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma Q(s', a') - Q(s, a)]$ |
| Q-learning | $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$ |
| Expected SARSA | $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \sum_{a'} \pi(a'|s') Q(s', a') - Q(s, a)]$ |

Reference: Sutton & Barto, *Reinforcement Learning: An Introduction*,
Example 6.6 (Cliff Walking).

## 1. Setup

In [ ]:
# cpai_lab パッケージのインストール（初回のみ）
!pip install -q git+https://github.com/oit-cpai/lab-core.git

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

# 可視化ヘルパーを cpai_lab からインポート
from cpai_lab.utils import smooth, plot_learning_curves, plot_cliff_policy

%matplotlib inline

SEED = 42
rng = np.random.default_rng(SEED)

env = gym.make('CliffWalking-v0')
print(f"Number of states : {env.observation_space.n}")
print(f"Number of actions: {env.action_space.n}")
print(f"Actions: 0=Up, 1=Right, 2=Down, 3=Left")

## 2. $\varepsilon$-greedy policy

The $\varepsilon$-greedy policy selects a random action with
probability $\varepsilon$ and the greedy action with probability
$1 - \varepsilon$.  When multiple actions share the maximum Q-value,
we break ties **randomly** to avoid directional bias.

In [ ]:
def epsilon_greedy_policy(Q_s, epsilon=0.1):
    '''Compute the epsilon-greedy action probabilities.

    Parameters
    ----------
    Q_s : ndarray, shape (n_actions,)
        Q-values for the current state.
    epsilon : float
        Exploration rate.

    Returns
    -------
    probs : ndarray, shape (n_actions,)
    '''
    n_actions = len(Q_s)

    # ── TODO 1 ──────────────────────────────────────────
    # 1) Initialize probs so that each action gets epsilon/n_actions.
    # 2) Find ALL actions whose Q-value equals the maximum (tie-break).
    # 3) Pick one of those best actions at random using rng.choice().
    # 4) Add (1 - epsilon) to that action's probability.
    # Return the probability array.
    # ────────────────────────────────────────────────────
    raise NotImplementedError("TODO 1: implement epsilon-greedy policy")

### Helper: sample an action

> **Note:** This function is provided. It uses your
> `epsilon_greedy_policy()` to sample one action.

In [ ]:
def sample_action(Q_s, epsilon=0.1):
    '''Sample one action from the epsilon-greedy distribution.'''
    probs = epsilon_greedy_policy(Q_s, epsilon)
    return rng.choice(len(probs), p=probs)

## 3. TD error

All three algorithms share the same update rule:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \, \delta$$
but differ in how the TD error $\delta$ is computed.

| Condition | TD error $\delta$ |
|-----------|-------------------|
| Terminal $s'$ | $r - Q(s, a)$ |
| **SARSA** | $r + \gamma \, Q(s', a') - Q(s, a)$ |
| **Q-learning** | $r + \gamma \, \max_{a'} Q(s', a') - Q(s, a)$ |
| **Expected SARSA** | $r + \gamma \, \sum_{a'} \pi(a'|s')\,Q(s',a') - Q(s, a)$ |

where $a'$ is the action actually taken at $s'$ (for SARSA) and
$\pi$ is the current $\varepsilon$-greedy policy (for Expected SARSA).

In [ ]:
def compute_td_error(method, Q, s, a, r, s_next, a_next,
                     terminated, epsilon=0.1, gamma=1.0):
    '''Compute the TD error for the given method.

    Parameters
    ----------
    method : str
        One of 'sarsa', 'qlearning', 'expected_sarsa'.
    Q : ndarray, shape (n_states, n_actions)
    s, a : int
        Current state and action.
    r : float
        Reward received.
    s_next, a_next : int
        Next state and next action (a_next used by SARSA only).
    terminated : bool
        Whether s_next is terminal.
    epsilon : float
        Exploration rate (used by Expected SARSA).
    gamma : float
        Discount factor.

    Returns
    -------
    td_error : float
    '''
    # ── TODO 2 ──────────────────────────────────────────
    # Implement the TD error calculation for each method.
    #
    # Hint:
    #   - If terminated: delta = r - Q[s, a]
    #   - SARSA: uses Q[s_next, a_next]
    #   - Q-learning: uses max over Q[s_next, :]
    #   - Expected SARSA: uses sum of pi(a'|s') * Q(s', a')
    #     where pi is the epsilon-greedy policy at s_next
    # ────────────────────────────────────────────────────
    raise NotImplementedError("TODO 2: implement TD error computation")

## 4. Training loop

In [ ]:
def run(method, alpha=0.5, epsilon=0.1, gamma=1.0,
        num_episodes=500, seed=None):
    '''Train an agent on CliffWalking using the specified method.

    Parameters
    ----------
    method : str
        'sarsa', 'qlearning', or 'expected_sarsa'.
    alpha : float
        Learning rate.
    epsilon : float
        Exploration rate for epsilon-greedy.
    gamma : float
        Discount factor.
    num_episodes : int
        Number of training episodes.
    seed : int or None
        Random seed for the environment.

    Returns
    -------
    Q : ndarray, shape (n_states, n_actions)
    rewards : ndarray, shape (num_episodes,)
        Total reward per episode.
    '''
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    rewards = np.zeros(num_episodes)

    for ep in range(num_episodes):
        s, _ = env.reset(seed=seed + ep if seed is not None else None)
        a = sample_action(Q[s, :], epsilon)

        terminated = False
        truncated = False

        # ── TODO 3 ──────────────────────────────────────
        # Write the episode loop:
        #   while not (terminated or truncated):
        #     1. Take action a -> get s_next, r, terminated, truncated
        #     2. Choose a_next from epsilon-greedy at s_next
        #     3. Compute TD error using compute_td_error()
        #     4. Update Q[s, a]
        #     5. s, a = s_next, a_next
        #     6. Accumulate reward
        # ────────────────────────────────────────────────
        raise NotImplementedError("TODO 3: implement the episode loop")

    return Q, rewards

## 5. Visualization

学習曲線の描画(移動平均 ±1 標準偏差)とポリシーの可視化(4×12グリッド上の矢印)には、
`cpai_lab.utils` の `plot_learning_curves` と `plot_cliff_policy` を使います(Setupセルでimport済み)。

- `plot_learning_curves(results, window=10, ylim=(-200, 0))` :
  `results = {手法名: shape (num_runs, num_episodes) の配列}` を渡す
- `plot_cliff_policy(Q, title=...)` : 学習済みQ値からgreedy方策を矢印で表示。
  SARSA(安全な経路)とQ-learning(崖沿いの最短経路)の違いがよく見える

## 6. Experiments

We compare all three algorithms with **identical hyperparameters**:
$\alpha = 0.5$, $\varepsilon = 0.1$, $\gamma = 1.0$, 500 episodes, 30 runs.

Using the same parameters is crucial for a fair comparison.

In [ ]:
# ── Hyperparameters ─────────────────────────────────────
ALPHA = 0.5
EPSILON = 0.1
GAMMA = 1.0
NUM_EPISODES = 500
NUM_RUNS = 30

In [ ]:
# ── TODO 4 ──────────────────────────────────────────
# Run all three algorithms (sarsa, qlearning, expected_sarsa)
# for NUM_RUNS runs each, and store the results in a dict:
#   results = {'SARSA': array(NUM_RUNS, NUM_EPISODES), ...}
#
# Use the run() function with the hyperparameters defined above.
# For reproducibility, pass seed=SEED + i * 1000 for run i.
# ────────────────────────────────────────────────────
raise NotImplementedError("TODO 4: run experiments for all three methods")

### 6.1 Learning curves

In [ ]:
plot_learning_curves(results, window=10, ylim=(-200, 0),
                     title='Cliff Walking: Learning Curves')

### 6.2 Learned policies

Observe:
- **SARSA** learns a **safe path** that stays away from the cliff,
  because during training the $\varepsilon$-greedy exploration
  occasionally steps onto the cliff — SARSA "knows" this and avoids it.
- **Q-learning** learns the **optimal (shortest) path** right along
  the cliff edge, because it updates toward the greedy max — ignoring
  the exploratory mistakes that happen during training.
- **Expected SARSA** typically behaves between the two.

In [ ]:
# Run one more time to get Q-values for visualization
q_sarsa, _    = run('sarsa',          alpha=ALPHA, epsilon=EPSILON,
                    gamma=GAMMA, num_episodes=NUM_EPISODES, seed=SEED)
q_qlearn, _   = run('qlearning',      alpha=ALPHA, epsilon=EPSILON,
                    gamma=GAMMA, num_episodes=NUM_EPISODES, seed=SEED)
q_expsarsa, _ = run('expected_sarsa',  alpha=ALPHA, epsilon=EPSILON,
                    gamma=GAMMA, num_episodes=NUM_EPISODES, seed=SEED)

plot_cliff_policy(q_sarsa,    title='SARSA Policy')
plot_cliff_policy(q_qlearn,   title='Q-learning Policy')
plot_cliff_policy(q_expsarsa, title='Expected SARSA Policy')

## 7. Discussion Questions

1. **Why does SARSA learn a safer (longer) path while Q-learning
   learns the optimal (cliff-edge) path?**
   Think about what each algorithm uses as the "target" in its TD update.

2. **What happens if you increase $\varepsilon$ (e.g., to 0.3)?**
   Does the gap between SARSA and Q-learning become larger or smaller?
   Why?

3. **Where does Expected SARSA fall relative to the other two?**
   Is it closer to SARSA or Q-learning? Why?

4. **What happens when $\varepsilon = 0$?**
   Do SARSA and Q-learning converge to the same policy?

## 8. Bonus: Hyperparameter Exploration

Try changing the hyperparameters below and re-running the experiments.

Suggested experiments:
- $\varepsilon \in \{0.01, 0.1, 0.3\}$ — How does exploration rate
  affect the learned policy?
- $\alpha \in \{0.1, 0.5, 0.9\}$ — How does the learning rate affect
  convergence speed and stability?

In [ ]:
# Try your own hyperparameters here!
# Example: compare epsilon = 0.01 vs 0.3

# eps_low_results = {}
# eps_high_results = {}
# for name, method in [('SARSA', 'sarsa'), ('Q-learning', 'qlearning')]:
#     data_low  = np.zeros((NUM_RUNS, NUM_EPISODES))
#     data_high = np.zeros((NUM_RUNS, NUM_EPISODES))
#     for i in range(NUM_RUNS):
#         _, data_low[i]  = run(method, alpha=ALPHA, epsilon=0.01,
#                               num_episodes=NUM_EPISODES, seed=SEED+i*1000)
#         _, data_high[i] = run(method, alpha=ALPHA, epsilon=0.3,
#                               num_episodes=NUM_EPISODES, seed=SEED+i*1000)
#     eps_low_results[f"{name} (eps=0.01)"] = data_low
#     eps_high_results[f"{name} (eps=0.3)"]  = data_high
# plot_learning_curves({**eps_low_results, **eps_high_results})